[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C31_Coding_Agent_Course/03_code_search/03_code_search.ipynb)

# 03 · 代码导航（agent 的眼）

目标：用**纯标准库（`re`/`glob`/`os.walk`/`ast`）** 从零写出编码 agent 的代码搜索——**grep**、**glob**、**智能遍历(滤 gitignore/二进制)**、**符号搜索(正则 + AST)**、**相关性排序 + 上下文行**——并在 `tempfile` 里建一个**真实的多文件玩具仓库**上搜、用 `assert` 验证。

路线：grep(含 re.escape) → glob → 智能遍历 → 符号搜索 → 排序+上下文 → 拼成搜索器→ ✏️ 练习 → 📖 答案 → 🧪 真实 stdlib 包胶囊。

> 心智模型：**agent 不能整库通读，靠搜索「看见」代码。grep 搜内容、glob 搜文件名；滤噪音、直达定义、排序——用最少 token 看见最相关代码。**

## 0 · 准备：一个真实的多文件玩具仓库

建一个有目录结构、有噪音（`__pycache__`、二进制）的小仓库，贴近真实代码库。

In [ ]:
import os, re, glob, ast, tempfile, shutil

WORK = tempfile.mkdtemp(prefix='c31_search_')
def seed(rel, content, binary=False):
    full = os.path.join(WORK, rel)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    mode = 'wb' if binary else 'w'
    with open(full, mode) as f:
        f.write(content)

seed('src/auth.py', 'def login(user, pw):\n    return check(user, pw)\n\n'
                    'def logout(user):\n    pass\n')
seed('src/db.py', 'def check(user, pw):\n    # login check\n    return user == "admin"\n')
seed('src/utils/helpers.py', 'def fmt(x):\n    return str(x)\n')
seed('tests/test_auth.py', 'from src.auth import login\n\n'
                           'def test_login():\n    assert login("admin", "x")\n')
seed('README.md', '# Demo\nCall login() to authenticate.\n')
# 噪音：依赖目录 + 二进制文件（应被智能遍历跳过）
seed('__pycache__/auth.cpython-311.pyc', b'\x00\x01login\x00binary', binary=True)
seed('node_modules/pkg/index.js', 'function login(){}\n')
print('多文件玩具仓库就绪 ✅  (含 src/、tests/、噪音目录)')

## 1 · grep：按正则在内容里逐行搜 ⭐

回答「`login` 在哪里出现过」。注意 **re.escape**：搜字面量时要转义，否则 `.`/`[` 等会被当正则元字符。

In [ ]:
def grep(work, pattern, files, literal=False):
    '''逐行搜 pattern。literal=True 时把 pattern 当字面量(自动 re.escape)。'''
    rx = re.compile(re.escape(pattern) if literal else pattern)
    hits = []
    for path in files:
        full = os.path.join(work, path)
        for i, line in enumerate(open(full, encoding='utf-8', errors='replace'), 1):
            if rx.search(line):
                hits.append((path, i, line.rstrip('\n')))
    return hits

py_files = ['src/auth.py', 'src/db.py', 'tests/test_auth.py']
hits = grep(WORK, r'\blogin\b', py_files)
for path, ln, text in hits:
    print(f'{path}:{ln}: {text.strip()}')
assert any(h[0] == 'src/auth.py' for h in hits), '应在 auth.py 命中 login'
assert any(h[0] == 'tests/test_auth.py' for h in hits)
# re.escape 的重要性：搜字面量 'a.b' 不应匹配 'axb'
seed('lit.py', 'axb = 1\na.b = 2\n')
lit_hits = grep(WORK, 'a.b', ['lit.py'], literal=True)
assert len(lit_hits) == 1 and lit_hits[0][1] == 2, 'literal 模式 a.b 只匹配字面 a.b'
print('✅ grep 正确；re.escape 让字面量搜不被特殊字符带偏')

## 2 · glob：按文件名/路径模式搜

回答「项目有哪些 .py 文件」。`**` 递归需 `recursive=True`。grep 搜内容、glob 搜文件名，互补。

In [ ]:
def glob_files(work, pattern):
    matches = glob.glob(os.path.join(work, pattern), recursive=True)
    return sorted(os.path.relpath(m, work) for m in matches if os.path.isfile(m))

all_py = glob_files(WORK, '**/*.py')
print('所有 .py:', all_py)
assert 'src/auth.py' in all_py and 'src/utils/helpers.py' in all_py
# 只看 src 下的（缩范围）
src_py = glob_files(WORK, 'src/**/*.py')
assert all(p.startswith('src/') for p in src_py)
assert 'tests/test_auth.py' not in src_py
# 找测试
tests = glob_files(WORK, '**/test_*.py')
assert tests == ['tests/test_auth.py']
print('✅ glob 正确：能按模式圈定文件范围（再交给 grep 搜内容）')

## 3 · 智能遍历：像 ripgrep 那样滤掉噪音

朴素遍历会被 `node_modules`/`__pycache__`/二进制淹没。智能遍历**剪枝**(不进忽略目录)+**过滤**(跳二进制)。
`dirs[:] = [...]` 原地改是 `os.walk` 的关键技巧。

In [ ]:
def is_binary(path, chunk=1024):
    with open(path, 'rb') as f:
        return b'\x00' in f.read(chunk)

def walk_code(work, ignore_dirs={'.git','__pycache__','node_modules','.venv','dist'}):
    out = []
    for root, dirs, files in os.walk(work):
        dirs[:] = [d for d in dirs if d not in ignore_dirs and not d.startswith('.')]  # 剪枝
        for fn in files:
            if fn.startswith('.'):
                continue
            full = os.path.join(root, fn)
            if is_binary(full):
                continue
            out.append(os.path.relpath(full, work))
    return sorted(out)

code_files = walk_code(WORK)
print('智能遍历结果:')
for f in code_files: print('  ', f)
# 噪音都被滤掉
assert not any('node_modules' in f for f in code_files), 'node_modules 应被剪枝'
assert not any('__pycache__' in f for f in code_files), '__pycache__ 应被剪枝'
assert not any(f.endswith('.pyc') for f in code_files), '二进制 .pyc 应被跳过'
assert 'src/auth.py' in code_files
print('✅ 智能遍历：依赖目录被剪枝、二进制被跳过，只剩值得搜的代码')

## 4 · 符号搜索：直达定义（正则 + AST）

grep `login` 返回所有出现；符号搜索只要**定义处**。先正则法（`def/class name`），再 AST 法（更精确）。

In [ ]:
def find_definition_regex(work, name, files):
    pat = re.compile(rf'^\s*(def|class)\s+{re.escape(name)}\b')
    hits = []
    for path in files:
        for i, line in enumerate(open(os.path.join(work, path), encoding='utf-8', errors='replace'), 1):
            if pat.search(line):
                hits.append((path, i, line.strip()))
    return hits

defs = find_definition_regex(WORK, 'login', walk_code(WORK))
print('login 的定义处(正则法):', defs)
assert defs == [('src/auth.py', 1, 'def login(user, pw):')], '只应命中定义，不含调用'

# AST 法：解析语法树，精确列出某文件所有函数/类及行号
def list_symbols_ast(work, path):
    tree = ast.parse(open(os.path.join(work, path), encoding='utf-8').read())
    syms = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            syms.append(('func', node.name, node.lineno))
        elif isinstance(node, ast.ClassDef):
            syms.append(('class', node.name, node.lineno))
    return sorted(syms, key=lambda s: s[2])

syms = list_symbols_ast(WORK, 'src/auth.py')
print('auth.py 的符号(AST 法):', syms)
assert ('func', 'login', 1) in syms and ('func', 'logout', 4) in syms
print('✅ 符号搜索：正则法直达定义，AST 法精确列出结构（更健壮）')

## 5 · 相关性排序 + 上下文行：把对的结果排前面

命中太多时，按相关性打分排序、只取前几条，并配上下文行（前后若干行），让 LLM 一眼看清。

In [ ]:
def score_hit(path, lineno, line, query):
    s = 0.0
    if re.search(rf'^\s*(def|class)\s+{re.escape(query)}\b', line): s += 10
    if query.lower() in os.path.basename(path).lower():               s += 3
    if '/test' in path or path.startswith('test') or 'docs/' in path or path.endswith('.md'): s -= 2
    s -= path.count('/') * 0.1
    return s

def context_lines(work, path, lineno, ctx=2):
    lines = open(os.path.join(work, path), encoding='utf-8', errors='replace').read().splitlines()
    lo, hi = max(0, lineno-1-ctx), min(len(lines), lineno+ctx)
    out = []
    for i in range(lo, hi):
        mark = '>' if i == lineno-1 else ' '
        out.append(f'{mark} {i+1:4d}| {lines[i]}')
    return '\n'.join(out)

all_files = walk_code(WORK)
hits = grep(WORK, 'login', all_files, literal=True)
ranked = sorted(hits, key=lambda h: score_hit(h[0], h[1], h[2], 'login'), reverse=True)
print('排序后第一条(应是定义处):', ranked[0][0], ranked[0][1])
assert ranked[0][0] == 'src/auth.py' and ranked[0][1] == 1, '定义处应排第一'
# README.md 里的命中应被降权排到后面
md_rank = [i for i,h in enumerate(ranked) if h[0]=='README.md'][0]
assert md_rank > 0, '文档命中应被降权'
print('\n第一条命中的上下文:')
print(context_lines(WORK, ranked[0][0], ranked[0][1]))
print('✅ 排序把定义处排第一、文档降权；上下文行让 LLM 看清周边')

## 6 · 拼成 agent 的「眼」：一个完整搜索器

建图→搜→排序→带上下文→截断，串成一个 `code_search`。它就是 agent 定位「该改哪」的入口。

In [ ]:
def code_search(work, query, top=5, ctx=2, literal=True):
    '''完整代码搜索：智能遍历 + grep + 排序 + 上下文 + 截断。返回对 LLM 友好的文本。'''
    files = walk_code(work)                                   # 滤噪音
    hits = grep(work, query, files, literal=literal)         # 搜内容
    ranked = sorted(hits, key=lambda h: score_hit(h[0], h[1], h[2], query), reverse=True)
    blocks = []
    for path, ln, _ in ranked[:top]:                         # 截断到前 top 条
        blocks.append(f'=== {path}:{ln} ===\n' + context_lines(work, path, ln, ctx))
    footer = '' if len(ranked) <= top else f'\n...(还有 {len(ranked)-top} 条命中未显示)'
    return (f'搜索 {query!r}：共 {len(ranked)} 条命中，显示最相关 {min(top,len(ranked))} 条\n\n'
            + '\n\n'.join(blocks) + footer)

result = code_search(WORK, 'login', top=3)
print(result)
assert 'src/auth.py:1' in result, '最相关的定义处应出现'
assert '共' in result and '命中' in result
print('\n✅ 完整搜索器跑通：一次调用给出排好序、带上下文、控了量的结果')

---
## ✏️ 练习 1：带「仅整词」「忽略大小写」选项的 grep

增强 grep：实现 `grep2(work, pattern, files, whole_word=False, ignore_case=False, literal=True)`。
- `literal` → 先 `re.escape`；
- `whole_word` → 用 `\b` 包住模式（只匹配完整词）；
- `ignore_case` → 编译时加 `re.IGNORECASE`。返回 `[(path, lineno, line), ...]`。

In [ ]:
def grep2(work, pattern, files, whole_word=False, ignore_case=False, literal=True):
    # TODO:
    #  pat = re.escape(pattern) if literal else pattern
    #  if whole_word: pat = r'\b' + pat + r'\b'
    #  flags = re.IGNORECASE if ignore_case else 0
    #  rx = re.compile(pat, flags); 逐行 rx.search 收集命中
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
seed('words.py', 'login = 1\nrelogin = 2\nLOGIN_URL = 3\nlogin_v2 = 4\n')
# 仅整词 login：只匹配第 1 行（不含 relogin/login_v2/LOGIN_URL）
ww = grep2(WORK, 'login', ['words.py'], whole_word=True)
assert [h[1] for h in ww] == [1], f'整词匹配应只命中第1行, got {[h[1] for h in ww]}'
# 忽略大小写 + 整词：再匹配 LOGIN_URL? 不——LOGIN_URL 是整词 'LOGIN_URL' 而非 'login'
ci = grep2(WORK, 'login', ['words.py'], whole_word=True, ignore_case=True)
assert 1 in [h[1] for h in ci]
# 不加整词、忽略大小写：login 子串到处都是
loose = grep2(WORK, 'login', ['words.py'], whole_word=False, ignore_case=True)
assert len(loose) >= 4
print('整词命中行:', [h[1] for h in ww], '| 宽松命中数:', len(loose))
print('✅ 练习 1 通过：whole_word / ignore_case / literal 选项都生效')

## ✏️ 练习 2：多模式 glob（include + exclude）

实现 `glob_filtered(work, include, exclude=())`：返回匹配 `include`（一个 glob 模式）、但**路径不含** `exclude` 里任一子串的文件。例如 `include='**/*.py', exclude=('test', '__pycache__')`。

In [ ]:
def glob_filtered(work, include, exclude=()):
    # TODO: files = glob_files(work, include)
    #       return [f for f in files if not any(x in f for x in exclude)]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
prod_py = glob_filtered(WORK, '**/*.py', exclude=('test', '__pycache__', 'node_modules', 'lit', 'words'))
print('生产代码 .py:', sorted(prod_py))
assert 'src/auth.py' in prod_py and 'src/db.py' in prod_py
assert not any('test' in f for f in prod_py), 'test 文件应被排除'
assert 'tests/test_auth.py' not in prod_py
# 不传 exclude 时等于普通 glob
assert set(glob_filtered(WORK, 'src/**/*.py')) == set(glob_files(WORK, 'src/**/*.py'))
print('✅ 练习 2 通过：include 圈范围、exclude 去噪音')

## ✏️ 练习 3：用 AST 跨文件找符号定义（含所在文件）

实现 `find_symbol_ast(work, name, files)`：用 **AST** 在多个文件里找名为 `name` 的函数或类定义，返回 `[(path, kind, lineno), ...]`（`kind` 为 `'func'` 或 `'class'`）。比正则法更可靠。

In [ ]:
def find_symbol_ast(work, name, files):
    results = []
    for path in files:
        # TODO: 解析 path 为 AST；ast.walk 找 FunctionDef/AsyncFunctionDef/ClassDef
        #       且 node.name == name；命中则 append (path, 'func'/'class', node.lineno)
        #       注意：源码可能有语法错误，用 try/except SyntaxError 跳过
        raise NotImplementedError
    return results

In [ ]:
# —— 练习 3 自测 ——
files = ['src/auth.py', 'src/db.py']
found = find_symbol_ast(WORK, 'check', files)
print('check 的定义:', found)
assert found == [('src/db.py', 'func', 1)], 'check 定义在 db.py 第1行'
# 找 login
assert find_symbol_ast(WORK, 'login', files) == [('src/auth.py', 'func', 1)]
# 不存在的符号
assert find_symbol_ast(WORK, 'nonexistent', files) == []
# 含语法错误的文件不应让整体崩溃
seed('broken.py', 'def oops(:\n  pass\n')
assert find_symbol_ast(WORK, 'check', files + ['broken.py']) == [('src/db.py', 'func', 1)]
print('✅ 练习 3 通过：AST 跨文件精确定位符号，且对语法错误健壮')

## ✏️ 练习 4：自定义相关性排序

实现 `rank_hits(hits, query, prefer_dir='src')`：给 grep 命中排序，规则（分数从高到低）：
定义处 +10；路径以 `prefer_dir` 开头 +3；文件名(basename)含 query +2；路径越深每层 -0.1。返回**排序后**的 hits 列表。

In [ ]:
def rank_hits(hits, query, prefer_dir='src'):
    def score(h):
        path, lineno, line = h
        s = 0.0
        # TODO: 按上面四条规则累加分数
        raise NotImplementedError
    return sorted(hits, key=score, reverse=True)

In [ ]:
# —— 练习 4 自测 ——
hits = grep(WORK, 'login', walk_code(WORK), literal=True)
ranked = rank_hits(hits, 'login', prefer_dir='src')
print('排序后前三:', [(h[0], h[1]) for h in ranked[:3]])
# 定义处(src/auth.py:1)分最高，应排第一
assert ranked[0][0] == 'src/auth.py' and ranked[0][1] == 1
# src/ 下的命中应整体靠前于 README.md / tests
src_positions = [i for i,h in enumerate(ranked) if h[0].startswith('src/')]
other_positions = [i for i,h in enumerate(ranked) if not h[0].startswith('src/')]
assert max(src_positions) < min(other_positions) if other_positions else True
print('✅ 练习 4 通过：定义处排首位、src/ 优先、按规则排序')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def grep2(work, pattern, files, whole_word=False, ignore_case=False, literal=True):
    pat = re.escape(pattern) if literal else pattern
    if whole_word:
        pat = r'\b' + pat + r'\b'
    rx = re.compile(pat, re.IGNORECASE if ignore_case else 0)
    hits = []
    for path in files:
        for i, line in enumerate(open(os.path.join(work, path), encoding='utf-8', errors='replace'), 1):
            if rx.search(line):
                hits.append((path, i, line.rstrip('\n')))
    return hits

In [ ]:
# 练习 2 参考答案
def glob_filtered(work, include, exclude=()):
    return [f for f in glob_files(work, include) if not any(x in f for x in exclude)]

In [ ]:
# 练习 3 参考答案
def find_symbol_ast(work, name, files):
    results = []
    for path in files:
        try:
            tree = ast.parse(open(os.path.join(work, path), encoding='utf-8').read())
        except SyntaxError:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
                results.append((path, 'func', node.lineno))
            elif isinstance(node, ast.ClassDef) and node.name == name:
                results.append((path, 'class', node.lineno))
    return results

In [ ]:
# 练习 4 参考答案
def rank_hits(hits, query, prefer_dir='src'):
    def score(h):
        path, lineno, line = h
        s = 0.0
        if re.search(rf'^\s*(def|class)\s+{re.escape(query)}\b', line): s += 10
        if path.startswith(prefer_dir): s += 3
        if query.lower() in os.path.basename(path).lower(): s += 2
        s -= path.count('/') * 0.1
        return s
    return sorted(hits, key=score, reverse=True)

---
## 🧪 真实数据胶囊：在一个真实 stdlib 包的源码树上搜索

用 Python **标准库里一个真实的包**（`json`，它是一个有多个 .py 文件的目录，随 Python 安装、真实存在）当样本，把整个包目录复制进工作区，用我们的搜索器**真实地**遍历它、grep 关键字、用 AST 列符号。

（无需联网：包就在你本机 Python 安装里，用 `os.path.dirname(json.__file__)` 定位。）

In [ ]:
import json as _json
pkg_dir = os.path.dirname(_json.__file__)          # 本机真实的 json 包目录
dst = os.path.join(WORK, 'real_json')
shutil.copytree(pkg_dir, dst,
                ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
rel_files = [os.path.relpath(os.path.join(r,f), WORK)
             for r,_,fs in os.walk(dst) for f in fs if f.endswith('.py')]
print(f'真实 json 包: {len(rel_files)} 个 .py 文件')
print('例:', sorted(os.path.basename(f) for f in rel_files)[:6])
# 智能遍历应能扫到这些真实源码、且不带 __pycache__
walked = [f for f in walk_code(WORK) if f.startswith('real_json/')]
assert len(walked) >= 3, '应扫到 json 包的多个源文件'
assert not any('__pycache__' in f for f in walked)
print('✅ 胶囊准备就绪：真实 stdlib 包已在工作区，可被搜索器遍历')

**🧪 胶囊练习**：在真实 `json` 包源码里，用 `code_search` 找出 `dumps` 这个公开函数的定义位置（它确实定义在 `json/__init__.py` 里），并用 `find_symbol_ast` 确认。补全骨架。

In [ ]:
json_files = [f for f in walk_code(WORK) if f.startswith('real_json/') and f.endswith('.py')]
# TODO:
#  1) result = code_search(WORK, 'dumps', top=5)   # 文本结果
#  2) defs = find_symbol_ast(WORK, 'dumps', json_files)   # [(path, 'func', lineno), ...]
raise NotImplementedError

In [ ]:
# 自测
assert 'dumps' in result
assert any(d[1] == 'func' and d[0].endswith('__init__.py') for d in defs), \
    'dumps 应定义在 json/__init__.py'
print('在真实 json 包里定位 dumps 定义:', [(os.path.basename(d[0]), d[2]) for d in defs])
print('✅ 胶囊练习通过：在真实 stdlib 源码树上完成了 遍历→搜索→符号定位 的闭环')

In [ ]:
# 📖 胶囊参考答案
result = code_search(WORK, 'dumps', top=5)
defs = find_symbol_ast(WORK, 'dumps', json_files)
print(result[:300])

---
## 🔧 旁注：搜索工具的 tool schema

把代码搜索暴露给真实 Claude 的 schema（**本环境不调用**）：

```python
SEARCH_TOOL_SCHEMA = {
    'name': 'code_search',
    'description': '在代码库里搜索一个字符串或符号，返回最相关的若干处命中（含文件:行号与上下文行）。'
                   '自动跳过 .git/node_modules/二进制等噪音。用它定位「该改哪个文件」。',
    'input_schema': {
        'type': 'object',
        'properties': {
            'query': {'type': 'string', 'description': '要搜的字符串或符号名，如 "login"'},
            'top':   {'type': 'integer', 'description': '最多返回几处命中，默认 5'},
        },
        'required': ['query'],
    },
}
```

`description` 里点明「自动跳噪音、用于定位该改哪」，等于把工具的**用法与价值**直接告诉模型。真实往返见模块 05。

In [ ]:
# 清理
shutil.rmtree(WORK, ignore_errors=True)
print('工作区已清理 ✅')

### 小结
- **grep / glob**：搜内容 / 搜文件名，正交互补；字面量搜务必 `re.escape`。
- **智能遍历**：剪枝忽略目录(`dirs[:]=...`) + 跳二进制(NUL 检测)，把领域常识编进工具。
- **符号搜索**：正则法直达定义、AST 法精确健壮——「文本近似 → 语法精确」的精度阶梯。
- **排序 + 上下文 + 截断**：给 LLM 当排序算法，用最少 token 呈现最相关代码——agent 的视力。
- **一句话**：agent 的智能一半在模型，一半在你喂它看的东西；搜索质量 = 视力。

下一站：**模块 04 · Edit-Test-Fix 循环** —— 把手、眼、脚组织成会自我纠错的小脑。